In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

In [ ]:
# Detectar e fixar o diretório raiz do projeto
# Sobe até encontrar a pasta que contém 'data/raw'
notebook_dir = Path().resolve()
print(f"Diretório atual: {notebook_dir}")

# Procurar o root do projeto subindo na hierarquia
project_root = notebook_dir
for _ in range(5):  # sobe até 5 níveis
    if (project_root / "data" / "raw").exists():
        break
    project_root = project_root.parent

print(f"Project root encontrado: {project_root}")
os.chdir(project_root)
print(f"Working directory fixado para: {Path().resolve()}")

# Paths
PHECODE12  = project_root / "data/raw/phecode12_long.txt"
PHECODEX   = project_root / "data/raw/phecodeX_long.txt"
COVARIATES = project_root / "data/raw/covariates.txt"

# Verificar se os arquivos existem
for name, path in [("PHECODE12", PHECODE12), ("PHECODEX", PHECODEX), ("COVARIATES", COVARIATES)]:
    status = "✅" if path.exists() else "❌"
    print(f"{status} {name}: {path}")

print(f"\nPandas version: {pd.__version__}")

Diretório atual: /project/hall/analysis/hearing-loss-genomics/notebooks/01_phase1_exploration
Project root encontrado: /project/hall/analysis/hearing-loss-genomics
Working directory fixado para: /project/hall/analysis/hearing-loss-genomics
✅ PHECODE12: /project/hall/analysis/hearing-loss-genomics/data/raw/phecode12_long.txt
✅ PHECODEX: /project/hall/analysis/hearing-loss-genomics/data/raw/phecodeX_long.txt
✅ COVARIATES: /project/hall/analysis/hearing-loss-genomics/data/raw/covariates.txt

Pandas version: 3.0.2


In [3]:
# Peek at first 5 rows without loading full file
df_peek = pd.read_csv(PHECODE12, sep="\t", nrows=5)
print("Shape preview (5 rows):", df_peek.shape)
print("\nColumn names:")
print(df_peek.columns.tolist())
print("\nFirst rows:")
df_peek

Shape preview (5 rows): (5, 5)

Column names:
['person_id', 'Phecode', 'num_visits', 'first_date', 'last_date']

First rows:


,person_id,Phecode,num_visits,first_date,last_date
0,PMBB1000274307312,208.00,2,2020-10-05,2020-10-05
1,PMBB1000274307312,211.00,3,2013-11-27,2017-04-30
2,PMBB1000274307312,244.20,7,2016-01-10,2022-09-17
3,PMBB1000274307312,244.40,25,2005-06-04,2020-07-26
4,PMBB1000274307312,250.42,15,2012-05-27,2021-03-02


In [ ]:
# Carregar apenas as colunas necessárias
df = pd.read_csv(PHECODE12, sep="\t", 
                 usecols=["person_id", "Phecode", "num_visits"],
                 dtype={"person_id": str, "Phecode": str})

# Filtrar phecode 389
# Atenção: coluna se chama 'Phecode' (P maiúsculo!)
hl_389 = df[df["Phecode"] == "389"]

print(f"Total participantes com phecode 389: {hl_389['person_id'].nunique():,}")
print(f"Total linhas:                        {len(hl_389):,}")
print(f"\nDistribuição de num_visits:")
print(hl_389["num_visits"].describe())
print(f"\nPrimeiras linhas:")
hl_389.head(10)

In [ ]:
# Classificar usando num_visits diretamente
cases    = hl_389[hl_389["num_visits"] >= 2]["person_id"]
excluded = hl_389[hl_389["num_visits"] == 1]["person_id"]

# Controles = todos os IDs que NÃO têm phecode 389
all_ids  = pd.Series(df["person_id"].unique())
controls = all_ids[~all_ids.isin(hl_389["person_id"])]

print("=== Phase 1 — Phecode 389 Classification ===")
print(f"Candidate cases   (num_visits >= 2): {len(cases):>8,}")
print(f"Excluded / NA     (num_visits == 1): {len(excluded):>8,}")
print(f"Potential controls (no phecode 389): {len(controls):>8,}")

In [ ]:
# Ver todos os phecodes que começam com 389
hl_389x = df[df["Phecode"].str.startswith("389", na=False)]
print("Variantes do phecode 389 encontradas:")
print(hl_389x["Phecode"].value_counts().head(20))

In [ ]:
# Efficient row count using chunking (file has 2.8M rows)
print("Counting rows and unique participants (this may take ~30s)...")

total_rows = 0
unique_ids = set()

for chunk in pd.read_csv(PHECODE12, sep="\t", chunksize=100_000, 
                          usecols=[0]):  # load only first column
    total_rows += len(chunk)
    unique_ids.update(chunk.iloc[:, 0].tolist())

print(f"Total rows:          {total_rows:,}")
print(f"Unique participants: {len(unique_ids):,}")

In [ ]:
# Load full file — only columns we need
print("Loading phecode file (filtering to phecode 389)...")

# First check column names from peek
id_col  = df_peek.columns[0]   # likely 'person_id'
phe_col = df_peek.columns[1]   # likely 'phecode' or similar

print(f"ID column:      {id_col}")
print(f"Phecode column: {phe_col}")

# Load efficiently
df = pd.read_csv(PHECODE12, sep="\t", usecols=[id_col, phe_col],
                 dtype={id_col: str, phe_col: str})

# Filter for hearing loss
hl = df[df[phe_col].str.startswith("389", na=False)]

print(f"\nTotal rows in file:        {len(df):,}")
print(f"Rows with phecode 389*:    {len(hl):,}")
print(f"Unique participants (389): {hl[id_col].nunique():,}")
print(f"\nPhecode 389 variants found:")
print(hl[phe_col].value_counts())

In [ ]:
# Count occurrences of phecode 389 per participant
hl_counts = hl.groupby(id_col).size().reset_index(name="phecode_389_count")

# Classify
cases    = hl_counts[hl_counts["phecode_389_count"] >= 2]
excluded = hl_counts[hl_counts["phecode_389_count"] == 1]

all_ids  = pd.Series(df[id_col].unique())
controls = all_ids[~all_ids.isin(hl_counts[id_col])]

print("=== Phase 1 — Phecode Classification ===")
print(f"Candidate cases   (>= 2 occurrences): {len(cases):>8,}")
print(f"Excluded / NA     (== 1 occurrence):  {len(excluded):>8,}")
print(f"Potential controls (0 occurrences):   {len(controls):>8,}")
print(f"Total accounted:                      {len(cases)+len(excluded)+len(controls):>8,}")
print(f"\nDistribution of occurrence counts (cases):")
print(hl_counts[hl_counts['phecode_389_count'] >= 2]['phecode_389_count'].describe())

In [ ]:
cov = pd.read_csv(COVARIATES, sep="\t")

print("=== Covariates File ===")
print(f"Shape: {cov.shape}")
print(f"\nColumns: {cov.columns.tolist()}")
print(f"\nData types:\n{cov.dtypes}")
print(f"\nFirst rows:")
cov.head()

In [ ]:
print("=== Open Questions Resolution ===\n")

# Q1: PC count
pc_cols = [c for c in cov.columns if c.upper().startswith("PC")]
print(f"Q1 - Number of PCs available: {len(pc_cols)}")
print(f"     PC columns: {pc_cols}")

# Q2: Age field
age_cols = [c for c in cov.columns if "age" in c.lower() or "birth" in c.lower() or "date" in c.lower() or "enroll" in c.lower()]
print(f"\nQ2 - Age-related columns: {age_cols}")

# Q3: Sex field
sex_cols = [c for c in cov.columns if "sex" in c.lower() or "gender" in c.lower()]
print(f"\nQ3 - Sex column: {sex_cols}")
print(f"\nSex distribution:")
if sex_cols:
    print(cov[sex_cols[0]].value_counts())